# Data Cleaning Project (pandas) - Messy Sales Data
**Goal:** clean a 500-row messy dataset (missing values, duplicates, formatting issues) and document every step.

**Run order:** Run every cell from top to bottom

## Step 1: Import libraries
pandas = cleaning, numpy = NaN values, re = text patterns, datetime = dates.

In [2]:
import re
import numpy as np
import pandas as pd
from datetime import datetime

## Step 2: Load the messy dataset
The dataset is loaded directly from my GitHub repository, so no file upload is needed.

In [3]:
url = "https://raw.githubusercontent.com/arooba2702/Data_Cleaning_Mini_Project/refs/heads/main/Raw_Sales_Data.csv"
raw = pd.read_csv(url)
print(raw.shape)
# Show the complete data as a scrollable, filterable table
from google.colab import data_table
data_table.enable_dataframe_formatter()
raw

(500, 13)


,Order ID,Customer Name,Email,Phone,Age,City,Product,Order Date,Quantity,Unit Price,Amount,Payment Method,Status
0,1333,Usman Zahra,USMAN.ZAHRA11@OUTLOOK.COM,0315 0766393,NaN,Karachi,keyboard,"January 05, 2026",1,73.41,NaN,COD,Shipped
1,1048,Noor Bhatti,NOOR.BHATTI45@HOTMAIL.COM,0345216-3449,23,Faisalabad,Lap top,18-06-2026,2,1252.46,"$2,504.92",Bank Transfer,cancelled
2,1248,Omar Hussain,omar.hussain92@yahoo.com,0344880-2632,unknown,faisalabad,monitor,2026-06-03,2,208.60,$417.20,cash,cancelled
3,1402,Nida Iqbal,NaN,0300-3649478,47,lahore,Key board,2026-05-07,1,41.30,Rs. 41,Cash,COMPLETED
4,1364,IMRAN ALI,imran.ali24@yahoo.com,(0344) 4366631,61,Rawalpindi,Laptop,2026-08-10,1,747.00,NaN,NaN,Returned
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,1042,FATIMA MALIK,FATIMA.MALIK90@HOTMAIL.COM,0333010-2438,27,FSD,headphones,2026-02-05,1,70.14,70.14,Credit Card,cancelled
496,1005,KAMRAN MALIK,kamran.malik70@outlook.com,0333819-7443,NaN,Islamabd,tablet,NaN,1,345.64,Rs. 346,JazzCash,cancelled
497,1382,Komal Javed,komal.javed24@mail.com,NaN,34,rawalpindi,PHONE,2026-02-21,1,754.69,$754.69,NaN,COMPLETED
498,1399,ALI ZAHRA,ali.zahra25@gmail.com,0333 1578878,34,Karchi,Headphones,NaN,2,103.89,208 USD,JazzCash,pending


## Step 3: Inspect the data
Always inspect before cleaning: size, types, missing values, duplicates, unique values.

In [4]:
print("Shape:", raw.shape)
print("\nData types:\n", raw.dtypes)
print("\nMissing values per column:\n", raw.isna().sum())
print("\nExact duplicate rows:", raw.duplicated().sum())
print("Repeated Order IDs  :", raw["Order ID"].duplicated().sum())
for c in ["City", "Product", "Payment Method", "Status"]:
    print(f"\n{c} values:\n", raw[c].value_counts(dropna=False).head(15))

Shape: (500, 13)

Data types:
 Order ID            int64
Customer Name      object
Email              object
Phone              object
Age                object
City               object
Product            object
Order Date         object
Quantity           object
Unit Price        float64
Amount             object
Payment Method     object
Status             object
dtype: object

Missing values per column:
 Order ID           0
Customer Name     23
Email             25
Phone             33
Age               24
City              29
Product           11
Order Date        30
Quantity          43
Unit Price        17
Amount            32
Payment Method    25
Status            16
dtype: int64

Exact duplicate rows: 14
Repeated Order IDs  : 30

City values:
 City
Faisalabad     39
faisalabad     34
NaN            29
Islamabad      26
rawalpindi     26
RWP            24
Rawalpindi     23
FSD            23
Karchi         20
lahore         19
ISB            18
Islamabd       18
karachi        

**What you should notice:** more repeated Order IDs than exact duplicate rows (near-duplicates hide behind case/spacing), many spellings of the same city/product, and text in numeric columns.

## Step 4: Working copy + cleaning log
Never edit the raw data. The log is your documentation of each step.

In [5]:
df = raw.copy()
log = []

def record(issue, action, before, after):
    log.append({"Step": len(log) + 1, "Issue found": issue, "Action taken": action,
                "Before": before, "After": after})

## Step 5: Standardize column names

In [6]:
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
record("Column names had spaces and capitals", "Converted to snake_case", "Order ID", "order_id")
list(df.columns)

['order_id',
 'customer_name',
 'email',
 'phone',
 'age',
 'city',
 'product',
 'order_date',
 'quantity',
 'unit_price',
 'amount',
 'payment_method',
 'status']

## Step 6: Remove empty rows
Rows that have only an Order ID carry no information.

In [7]:
before = len(df)
data_cols = [c for c in df.columns if c != "order_id"]
df = df.dropna(how="all", subset=data_cols)
record("Rows with an Order ID but no other data", "Dropped them",
       f"{before} rows", f"{len(df)} rows")
len(df)

495

## Step 7: Trim spaces and fix capitalization (names, email)

In [8]:
text_cols = ["customer_name", "email", "phone", "city", "product", "payment_method", "status"]
for c in text_cols:
    df[c] = df[c].astype("string").str.strip()

df["customer_name"] = (df["customer_name"].str.replace(r"\s+", " ", regex=True).str.title())
record("Names had extra spaces and mixed case", "Trimmed, collapsed spaces, Title Case",
       "inconsistent", "consistent")
df["customer_name"].head()

,customer_name
0,Usman Zahra
1,Noor Bhatti
2,Omar Hussain
3,Nida Iqbal
4,Imran Ali


## Step 8: Clean and validate emails
Remove stray spaces and lowercase. Invalid emails (no `@` or no `.com`) become missing. We do not guess a fix.

In [9]:
df["email"] = df["email"].str.replace(r"\s+", "", regex=True).str.lower()
valid = df["email"].str.match(r"^[\w.\-]+@[\w\-]+\.[a-z]{2,}$", na=False)
bad = int((~valid & df["email"].notna()).sum())
df.loc[~valid, "email"] = pd.NA
record(f"{bad} invalid emails (missing @ or domain extension), plus case/space issues",
       "Removed spaces, lowercased, set invalid to missing", f"{bad} invalid", "0 invalid")
df["email"].head()

,email
0,usman.zahra11@outlook.com
1,noor.bhatti45@hotmail.com
2,omar.hussain92@yahoo.com
3,<NA>
4,imran.ali24@yahoo.com


## Step 9: Standardize phone numbers
Keep digits only. `+92300...` becomes `0300...`. Valid = 11 digits, shown as `0XXX-XXXXXXX`. Too-short numbers become missing.

In [10]:
def fmt_phone(p):
    if pd.isna(p):
        return pd.NA
    d = re.sub(r"\D", "", str(p))
    if d.startswith("92") and len(d) == 12:
        d = "0" + d[2:]
    return f"{d[:4]}-{d[4:]}" if len(d) == 11 else pd.NA

before_bad = int(df["phone"].notna().sum())
df["phone"] = df["phone"].apply(fmt_phone)
record("Phones in many formats (dashes, spaces, +92, brackets) and some too short",
       "Digits only, +92 converted, reformatted to 0XXX-XXXXXXX, invalid set to missing",
       f"{before_bad} filled", f"{int(df['phone'].notna().sum())} valid")
df["phone"].head()

,phone
0,0315-0766393
1,0345-2163449
2,0344-8802632
3,0300-3649478
4,0344-4366631


## Step 10: Fix Age
Convert to numeric (text like `N/A`, `thirty` becomes missing), then treat ages outside 16-100 as invalid.

In [11]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")
mask = (df["age"] < 16) | (df["age"] > 100)
n_bad = int(mask.sum())
df.loc[mask, "age"] = np.nan
record(f"Age had text and {n_bad} impossible values (250, 999, 0, -5...)",
       "Converted to numeric; values outside 16-100 set to missing (filled later)",
       f"{n_bad} impossible", "0 impossible")
df["age"].describe()

,age
count,457.000000
mean,42.218818
std,13.809966
min,18.000000
25%,30.000000
50%,43.000000
75%,54.000000
max,65.000000


## Step 11: Standardize city, product, payment method, status
Dictionaries map every variant to one standard value.

In [12]:
city_map = {"lahore": "Lahore", "lhr": "Lahore", "lahor": "Lahore",
            "karachi": "Karachi", "khi": "Karachi", "karchi": "Karachi",
            "islamabad": "Islamabad", "isb": "Islamabad", "islamabd": "Islamabad",
            "rawalpindi": "Rawalpindi", "rwp": "Rawalpindi",
            "faisalabad": "Faisalabad", "fsd": "Faisalabad"}
product_map = {"laptop": "Laptop", "lap top": "Laptop",
               "phone": "Phone", "mobile": "Phone",
               "tablet": "Tablet", "tab": "Tablet",
               "monitor": "Monitor",
               "keyboard": "Keyboard", "key board": "Keyboard",
               "mouse": "Mouse",
               "headphones": "Headphones", "head phones": "Headphones"}
payment_map = {"credit card": "Credit Card", "cc": "Credit Card",
               "cash": "Cash",
               "cash on delivery": "Cash on Delivery", "cod": "Cash on Delivery",
               "jazzcash": "JazzCash", "jazz cash": "JazzCash",
               "easypaisa": "Easypaisa", "bank transfer": "Bank Transfer"}
status_map = {"completed": "Completed", "pending": "Pending", "shipped": "Shipped",
              "cancelled": "Cancelled", "canceled": "Cancelled", "returned": "Returned"}

def standardize(series, mapping):
    key = series.str.lower().str.strip()
    return key.map(mapping).fillna(series)

for col, mp in [("city", city_map), ("product", product_map),
                ("payment_method", payment_map), ("status", status_map)]:
    n_before = raw[col.replace("_", " ").title()].nunique()
    df[col] = standardize(df[col], mp)
    record(f"{col} had spelling/case variants", "Mapped all variants to one standard value",
           f"{n_before} unique values", f"{df[col].nunique()} unique values")

for c in ["city", "product", "payment_method", "status"]:
    print(c, "->", df[c].value_counts(dropna=False).to_dict())

city -> {'Islamabad': 100, 'Faisalabad': 96, 'Karachi': 92, 'Lahore': 92, 'Rawalpindi': 91, <NA>: 24}
product -> {'Laptop': 87, 'Monitor': 85, 'Phone': 77, 'Keyboard': 67, 'Mouse': 63, 'Headphones': 59, 'Tablet': 51, <NA>: 6}
payment_method -> {'Credit Card': 135, 'JazzCash': 82, 'Cash': 80, 'Easypaisa': 70, 'Cash on Delivery': 69, 'Bank Transfer': 39, <NA>: 20}
status -> {'Cancelled': 134, 'Completed': 133, 'Shipped': 96, 'Pending': 79, 'Returned': 42, <NA>: 11}


## Step 12: Fix dates
Several formats are parsed one by one. Day comes before month (dd/mm/yyyy). This is an assumption, so state it in your report. Invalid dates (`31/02/2026`, `N/A`) become missing.

In [13]:
FORMATS = ("%Y-%m-%d", "%d/%m/%Y", "%b %d, %Y", "%d-%m-%Y", "%B %d, %Y")

def parse_date(x):
    if pd.isna(x):
        return pd.NaT
    for fmt in FORMATS:
        try:
            return datetime.strptime(str(x).strip(), fmt)
        except ValueError:
            continue
    return pd.NaT

n_before = int(df["order_date"].notna().sum())
df["order_date"] = pd.to_datetime(df["order_date"].apply(parse_date))
record("5 date formats plus invalid dates", "Parsed to real dates (dd/mm/yyyy assumed); invalid set to missing",
       f"{n_before} text values", f"{int(df['order_date'].notna().sum())} valid dates")
df["order_date"].describe()

,order_date
count,467
mean,2026-05-16 04:15:55.888651008
min,2026-01-02 00:00:00
25%,2026-03-11 00:00:00
50%,2026-05-17 00:00:00
75%,2026-07-16 12:00:00
max,2026-09-27 00:00:00


## Step 13: Fix Quantity, Unit Price and Amount
- **Quantity:** words like `two` become 2; values below 1 or above 20 are invalid.
- **Amount:** remove `$`, `Rs.`, `USD` and commas, then convert to a number.
- **Unit price:** a price above 3x the median for its product is treated as an outlier.

In [14]:
# Quantity
words = {"one": "1", "two": "2", "three": "3", "four": "4", "five": "5"}
q = df["quantity"].astype("string").str.strip().str.lower().replace(words)
df["quantity"] = pd.to_numeric(q, errors="coerce")
bad_q = (df["quantity"] < 1) | (df["quantity"] > 20)
n_q = int(bad_q.sum())
df.loc[bad_q, "quantity"] = np.nan
record(f"Quantity had words and {n_q} impossible values (-1, 0, 100)",
       "Words converted; invalid set to missing", f"{n_q} invalid", "0 invalid")

# Amount
amt = (df["amount"].astype("string").str.lower()
       .str.replace(r"rs\.?|usd|\$|,", "", regex=True).str.strip())
df["amount"] = pd.to_numeric(amt, errors="coerce")
record("Amount stored as text ($, Rs., USD, commas)", "Stripped symbols, converted to float",
       "text", "numeric")

# Unit price outliers
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
med = df.groupby("product")["unit_price"].transform("median")
out = df["unit_price"] > 3 * med
n_out = int(out.sum())
df.loc[out, "unit_price"] = np.nan
record(f"{n_out} unit prices more than 3x their product median", "Set to missing (filled later)",
       f"{n_out} outliers", "0 outliers")
df[["quantity", "unit_price", "amount"]].describe()

,quantity,unit_price,amount
count,451.0,476.000000,468.0
mean,1.784922,384.531786,697.791047
std,1.026469,371.111034,855.681871
min,1.0,8.700000,13.64
25%,1.0,66.687500,94.51
50%,1.0,284.455000,396.0
75%,2.0,577.792500,998.915
max,5.0,1392.560000,5536.0


## Step 14: Remove duplicates
This comes after standardizing. Before that, near-duplicates still look different. Keep the first record per Order ID.

In [15]:
before = len(df)
df = df.drop_duplicates(subset=["order_id"], keep="first")
record("Repeated Order IDs (exact copies and near-copies)", "Dropped repeats, kept first",
       f"{before} rows", f"{len(df)} rows")
print("Rows now:", len(df), "| repeated IDs left:", df["order_id"].duplicated().sum())

Rows now: 466 | repeated IDs left: 0


## Step 15: Handle missing values
Order of preference: 1) calculate from related columns (`amount = quantity x unit_price`), 2) group median, 3) label as `Unknown`.

In [16]:
n_miss = df[["age", "quantity", "unit_price", "amount"]].isna().sum().to_dict()

# 1) derive from related columns
m = df["amount"].isna() & df["quantity"].notna() & df["unit_price"].notna()
df.loc[m, "amount"] = df["quantity"] * df["unit_price"]
m = df["unit_price"].isna() & df["amount"].notna() & df["quantity"].notna()
df.loc[m, "unit_price"] = df["amount"] / df["quantity"]
m = df["quantity"].isna() & df["amount"].notna() & df["unit_price"].notna()
df.loc[m, "quantity"] = (df["amount"] / df["unit_price"]).round()

# 2) group median / overall fallback
df["unit_price"] = df["unit_price"].fillna(df.groupby("product")["unit_price"].transform("median"))
df["unit_price"] = df["unit_price"].fillna(df["unit_price"].median())
df["quantity"] = df["quantity"].fillna(df["quantity"].mode()[0])
df["amount"] = df["amount"].fillna(df["quantity"] * df["unit_price"])
age_med = df["age"].median()
df["age"] = df["age"].fillna(age_med)

# 3) labels for categories / contacts
for c in ["customer_name", "city", "product", "payment_method", "status"]:
    df[c] = df[c].fillna("Unknown")
for c in ["email", "phone"]:
    df[c] = df[c].fillna("Not provided")

record(f"Missing values: {n_miss}",
       f"amount/price/quantity derived from each other, then median; age = median ({age_med:.0f}); "
       "text fields = 'Unknown' / 'Not provided'; order_date left blank (cannot be invented)",
       str(n_miss), "0 (except order_date)")
df.isna().sum()

,0
order_id,0
customer_name,0
email,0
phone,0
age,0
city,0
product,0
order_date,25
quantity,0
unit_price,0


## Step 16: Final formatting and validation

In [17]:
df["age"] = df["age"].astype(int)
df["quantity"] = df["quantity"].astype(int)
df["unit_price"] = df["unit_price"].round(2)
df["amount"] = df["amount"].round(2)
df = df.sort_values("order_id").reset_index(drop=True)

final = df.copy()
final["order_date"] = final["order_date"].dt.strftime("%Y-%m-%d")
final.columns = [c.replace("_", " ").title().replace("Id", "ID") for c in final.columns]

assert not final["Order ID"].duplicated().any(), "Duplicates remain!"
assert final.drop(columns="Order Date").isna().sum().sum() == 0, "Nulls remain!"
record("Final checks", "Sorted by Order ID, rounded money, validated no duplicates/nulls",
       f"{len(raw)} raw rows", f"{len(final)} clean rows")
print("Validation passed.")
final.head(15)

Validation passed.


,Order ID,Customer Name,Email,Phone,Age,City,Product,Order Date,Quantity,Unit Price,Amount,Payment Method,Status
0,1001,Danish Malik,danish.malik72@yahoo.com,0344-7038374,55,Unknown,Laptop,2026-03-23,1,1219.09,1219.0,Credit Card,Shipped
1,1002,Hina Tariq,hina.tariq74@yahoo.com,0300-3823498,23,Faisalabad,Monitor,2026-07-14,3,151.92,456.0,Cash on Delivery,Shipped
2,1003,Zain Iqbal,zain.iqbal30@gmail.com,0345-4491946,44,Faisalabad,Mouse,2026-04-19,2,47.34,94.68,Easypaisa,Pending
3,1004,Bilal Sheikh,bilal.sheikh88@mail.com,0345-6402509,47,Lahore,Mouse,2026-01-06,1,31.58,31.58,JazzCash,Returned
4,1005,Kamran Malik,kamran.malik70@outlook.com,0333-8197443,44,Islamabad,Tablet,NaN,1,345.64,346.0,JazzCash,Cancelled
5,1006,Saad Raza,saad.raza57@outlook.com,0312-3769795,44,Karachi,Phone,NaN,4,678.84,2715.0,Credit Card,Completed
6,1007,Amna Butt,amna.butt55@mail.com,0300-1651177,44,Rawalpindi,Laptop,2026-05-08,1,1059.91,1060.0,Credit Card,Pending
7,1008,Omar Sheikh,omar.sheikh8@yahoo.com,0345-4449368,47,Karachi,Headphones,2026-09-07,1,186.19,186.0,Credit Card,Cancelled
8,1009,Sara Butt,sara.butt73@yahoo.com,0300-1375453,55,Lahore,Laptop,2026-05-14,1,735.10,735.0,Cash,Completed
9,1010,Noor Farooq,noor.farooq48@hotmail.com,0344-5075395,59,Islamabad,Monitor,2026-06-03,2,224.34,449.0,Credit Card,Completed


## Step 17: Before vs after summary

In [18]:
summary = pd.DataFrame({
    "Raw": [len(raw), int(raw.isna().sum().sum()), int(raw["Order ID"].duplicated().sum())],
    "Cleaned": [len(final), int(final.isna().sum().sum()), int(final["Order ID"].duplicated().sum())]
}, index=["Rows", "Missing cells", "Repeated Order IDs"])
display(summary)

log_df = pd.DataFrame(log)
pd.set_option("display.max_colwidth", 90)
display(log_df)

,Raw,Cleaned
Rows,500,466
Missing cells,308,25
Repeated Order IDs,30,0


,Step,Issue found,Action taken,Before,After
0,1,Column names had spaces and capitals,Converted to snake_case,Order ID,order_id
1,2,Rows with an Order ID but no other data,Dropped them,500 rows,495 rows
2,3,Names had extra spaces and mixed case,"Trimmed, collapsed spaces, Title Case",inconsistent,consistent
3,4,"26 invalid emails (missing @ or domain extension), plus case/space issues","Removed spaces, lowercased, set invalid to missing",26 invalid,0 invalid
4,5,"Phones in many formats (dashes, spaces, +92, brackets) and some too short","Digits only, +92 converted, reformatted to 0XXX-XXXXXXX, invalid set to missing",467 filled,450 valid
5,6,"Age had text and 12 impossible values (250, 999, 0, -5...)",Converted to numeric; values outside 16-100 set to missing (filled later),12 impossible,0 impossible
6,7,city had spelling/case variants,Mapped all variants to one standard value,31 unique values,5 unique values
7,8,product had spelling/case variants,Mapped all variants to one standard value,23 unique values,7 unique values
8,9,payment_method had spelling/case variants,Mapped all variants to one standard value,12 unique values,6 unique values
9,10,status had spelling/case variants,Mapped all variants to one standard value,13 unique values,5 unique values


## Step 18: Export and download
Colab files are temporary. Download them now.

In [19]:
from google.colab import files
with pd.ExcelWriter("Cleaned_Dataset.xlsx", engine="openpyxl") as xw:
    final.to_excel(xw, sheet_name="Cleaned_Data", index=False)
    raw.to_excel(xw, sheet_name="Raw_Data", index=False)
    log_df.to_excel(xw, sheet_name="Cleaning_Log", index=False)

final.to_csv("Cleaned_Data.csv", index=False)
files.download("Cleaned_Dataset.xlsx")
files.download("Cleaned_Data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes for the report
- Dates are read as day/month/year.
- Ages outside 16-100 and quantities outside 1-20 are treated as invalid.
- Invalid emails and phones are set to missing and not guessed.
- `Rs.` amounts were treated as plain numbers (no currency conversion).
- Missing order dates are left blank because they cannot be reliably inferred.